# 作业：英中翻译任务


在本notebook中，您将构建一个Transformer模型（神经网络模型）来实现将英文句子翻译成中文句子。

参考论文：[Attention Is All You Need](https://arxiv.org/pdf/1706.03762.pdf)

![](document/images/google_translate.png )

完整数据集包含约10,000个句子对。但这对于提高您的Python编程技能和更深入理解PyTorch与Transformer（神经网络）是非常好的练习。

![Attention Is All You Need](document/images/google_paper.png)

## 1. 理解Transformer模型
 
整个Transformer编码器-解码器模型架构服务于以下目的：


- 编码器（Encoder）：编码过程将输入句子（英文单词列表）转换为数值矩阵格式（嵌入向量）。可以将这个步骤视为从输入中提取对解码器有用和必要的信息。在图06中，嵌入向量由绿色矩阵表示。

- 解码器（Decoder）：然后解码过程将这些嵌入向量映射回另一种语言序列，如图06所示。这有助于我们解决各种监督式NLP任务，如机器翻译（本例）、情感分类、实体识别、摘要生成、语义关系提取等。
 
![Understand_How_Transformer_Work](./document/images/Understand_How_Transformer_Work.png)

 

## 2. 编码器
  
在本节中，我们将重点介绍编码器的结构，因为理解了编码器的结构后，理解解码器就会变得非常简单。此外，我们还可以仅使用编码器来完成NLP中的一些主流任务，如情感分类、语义关系分析、命名实体识别等。

回顾一下，编码器表示将自然语言序列映射到隐藏层输出的数学表达式的过程。

**这是Transformer编码器块的结构**
> 注意：以下部分将参考1、2、3、4块。

![Transformer Encoder Stacks](./document/images/encoder.png)

### 2.0 数据准备：英中翻译器数据

In [ ]:
import os
import math
import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from nltk import word_tokenize
from collections import Counter
from torch.autograd import Variable
import seaborn as sns
import matplotlib.pyplot as plt

: 

In [ ]:
# 首次使用时，您可能需要下载nltk包的以下数据包
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

#### 输入/输出嵌入（Input/Output Embeddings）
与所有序列模型类似，我们使用学习到的嵌入层将输入/输出向量的维度转换为$d_{model}$。
在我们的模型中，两个嵌入层和pre-softmax层共享权重矩阵。

In [ ]:
# 创建输入嵌入层
class InputEmbeddings(nn.Module):
    
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model # 向量维度（512）
        self.vocab_size = vocab_size # 词表大小
        self.embedding = nn.Embedding(vocab_size, d_model) # PyTorch层，将整数索引转换为稠密嵌入向量
        
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model) # 归一化嵌入向量的方差

: 

现在，我们已经完成了所有数据预处理的代码。让我们专注于理解和构建Transformer模型。 

### 2.1 位置编码（Positional Encoding）

Transformer在编码器中**不**包含像RNN或LSTM这样的循环操作，因此我们必须向模型提供单词的位置信息，以便模型学习输入序列中的顺序。

因此，我们定义位置编码为[max_sequence_length, embedding_dimension]

在论文中，我们使用正弦和余弦函数来提供位置信息。

$$PE_{(pos,2i)} = sin(pos / 10000^{2i/d_{\text{model}}}) \quad \quad PE_{(pos,2i+1)} = cos(pos / 10000^{2i/d_{\text{model}}})\tag{公式1}$$  

In [ ]:
# 创建位置编码
class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model # 模型维度
        self.seq_len = seq_len # 最大序列长度
        self.dropout = nn.Dropout(dropout) # Dropout层，防止过拟合
        
        # 创建一个形状为(seq_len, d_model)的位置编码矩阵，初始化为0
        pe = torch.zeros(seq_len, d_model) 
        
        # 创建一个表示位置的张量（0到seq_len-1）
        position = torch.arange(0, seq_len, dtype = torch.float).unsqueeze(1) # 将'position'转换为2D张量[seq_len, 1]
        
        # 创建位置编码公式中的除法项
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # 对pe中的偶数索引应用正弦函数
        pe[:, 0::2] = torch.sin(position * div_term)
        # 对pe中的奇数索引应用余弦函数
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 在pe矩阵前面添加一个维度，用于批次处理
        pe = pe.unsqueeze(0)
        
        # 将'pe'注册为buffer。Buffer是不被视为模型参数的张量
        self.register_buffer('pe', pe) 
        
    def forward(self,x):
        # 将位置编码添加到输入张量X
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x) # Dropout正则化

我们可以看到，我们首先基于x构建位置编码，然后在forward函数中将'pe'加到x上。

> 注意：设置'requires_grad=False'，因为我们不需要训练pe。

下面是位置嵌入的可视化图，您可以看到随着嵌入维度增加，模式是如何变化的。 

In [ ]:
pe = PositionalEncoding(32, 100, 0)  # d_model, dropout比例, 最大长度
positional_encoding = pe.forward(torch.zeros(1, 100, 32))  # 序列长度, d_model
plt.figure(figsize=(10, 10))
sns.heatmap(positional_encoding.squeeze())  # 100x32矩阵
plt.title("正弦函数")
plt.xlabel("隐藏维度")
plt.ylabel("序列长度")


plt.figure(figsize=(15, 5))
pe = PositionalEncoding(24, 100, 0)
y = pe.forward(torch.zeros(1, 100, 24))
plt.plot(np.arange(100), y[0, :, 5:10].data.numpy())
plt.legend(["dim %d" % p for p in [5, 6, 7, 8, 9]])

### 2.2 自注意力机制与掩码（Self Attention and Mask）

注意力函数可以描述为将一个查询（query）和一组键值对（key-value pairs）映射到一个输出的过程，其中查询、键、值和输出都是向量。输出被计算为值的加权和，其中分配给每个值的权重由查询与相应键的兼容性函数计算得出。

我们称我们的特定注意力为"缩放点积注意力（Scaled Dot-Product Attention）"。输入由维度为$d_k$的查询和键，以及维度为$d_v$的值组成。

我们计算查询与所有键的点积，除以$\sqrt{d_k}$，然后应用softmax函数来获得值的权重。

![self_attention](document/images/self-attention.png)


两种最常用的注意力函数是加法注意力和点积（乘法）注意力。点积注意力与我们的算法相同，只是多了一个$\frac{1}{\sqrt{d_k}}$的缩放因子。加法注意力使用具有单个隐藏层的前馈网络计算兼容性函数。虽然两者在理论复杂度上相似，但点积注意力在实践中更快、更节省空间，因为它可以使用高度优化的矩阵乘法代码实现。

为了解释为什么点积会变大，假设q和k的分量是均值为0方差为1的独立随机变量。那么它们的点积$q\cdot k$的均值为0，方差为$d_k$。为了抵消这种影响，我们将点积除以$\frac{1}{\sqrt{d_k}}$进行缩放。

多头注意力（Multi-head attention）允许模型在不同的表示子空间中共同关注不同位置的信息。如果只有一个注意力头，平均操作会抑制这种能力。

**注意力掩码（Attention Mask）**

输入$X$是$[batch-size,  sequence-length]$，我们使用'padding'用0填充矩阵，以最长的序列为准。

但这会导致softmax计算的问题。
$\sigma(\mathbf {z})_{i}={\frac {e^{z_{i}}}{\sum _{j=1}^{K}e^{z_{j}}}}$，其中$e^0=1$。

这意味着padding部分也参与计算，但它们不应该参与。因此，我们创建这个掩码来通过赋予大负偏置来忽略这些区域。
 
$$z_{illegal} = z_{illegal} + bias_{illegal}$$
$$bias_{illegal} \to -\infty$$
$$e^{z_{illegal}} \to 0 $$

因此，掩码区域的输出将变为0，从而避免它们参与计算。

> 注意：在自注意力计算中，我们使用小批量数据作为输入，意味着我们向模型输入多行句子进行训练和计算。


![](document/images/attention_mask.png)

在Transformer中，编码器和解码器的注意力计算都需要掩码操作，但它们的功能不同。

在解码器中，自注意力层只允许关注输出序列中较早的位置。这是通过在自注意力计算中的softmax步骤之前屏蔽（设置为'-inf'）未来位置来实现的。

"编码器-解码器注意力"层的工作方式与多头自注意力相同，只是它的查询矩阵来自它下面的层，而键和值矩阵来自编码器堆栈的输出。

在这里，我们定义一个批处理对象，用于保存训练时的源句（英文）和目标句（中文），以及构建掩码。

> 注意：掩码（可选）位于Scale和Softmax之间。

In [ ]:
class MaskBatch:
    '''用于在训练时保存一批数据和掩码的对象'''
    def __init__(self, src, tgt=None, pad_id=0):
        # 将单词id转换为long格式
        src = torch.from_numpy(src).long()
        tgt = torch.from_numpy(tgt).long()
        self.src = src
        # 获取padding位置的二元掩码
        self.src_mask = (src != pad_id).unsqueeze(1).unsqueeze(1) # 掩码 [B 1 1 src_L]
        if tgt is not None:
            # 解码器输入
            self.tgt = tgt[:, :-1]
            # 解码器目标
            self.tgt_y = tgt[:, 1:]
            # 为解码器输入添加注意力掩码
            self.tgt_mask = self.make_decoder_mask(self.tgt, pad_id)
            # 检查解码器输出padding数量
            self.ntokens = (self.tgt_y != pad_id).data.sum()

    def make_decoder_mask(self, tgt, pad_id):
        "创建用于隐藏padding和未来词的掩码"
        tgt_mask = (tgt != pad_id).unsqueeze(1).unsqueeze(1) # 掩码 [B 1 1 tgt_L] 
        tgt_mask = tgt_mask & casual_mask(tgt.size(-1)).unsqueeze(1).type_as(tgt_mask.data) 
        return tgt_mask 

### 2.3 层归一化与残差连接（Layer Normalization and Residual Connection）

1). **层归一化（LayerNorm）**：

**层归一化**将隐藏层输出标准化为标准格式，i.i.d，以提高训练效率和模型权重收敛（按行）。
$$\mu_{i}=\frac{1}{m} \sum^{m}_{j=1}x_{ij}$$

$$\sigma^{2}_{i}=\frac{1}{m} \sum^{m}_{j=1}
(x_{ij}-\mu_{i})^{2}$$

$$LayerNorm(x)=\alpha \odot \frac{x_{ij}-\mu_{i}}
{\sqrt{\sigma^{2}_{i}+\epsilon}} + \beta \tag{公式5}$$

$\epsilon$是为了避免除以0；**$\alpha, \beta$**是参数，$\odot$表示逐元素乘积。通常，我们初始化$\alpha$为1，$\beta$为0。

2). **残差连接（Residual Connection）**：

我们在两个子层的每一个周围使用残差连接，然后进行层归一化。
我们从注意力$Attention(Q,  K, V)$的权重中获得值矩阵，然后将其转置以确保与$X_{embedding}:[batch.size, sequence.length, embedding.dimension]$共享相同的形状。然后将它们相加。

$$X_{embedding} + Attention(Q, K, V)$$

在下面的计算中，在每个模块之后，我们添加输入与模块的输出以获得残差连接，这允许梯度反向传播到起始层。
$$X + SubLayer(X) \tag{公式6}$$

> **注意**：对于$SubLayer(X)$，我们调用dropout函数然后添加x，即$X + Dropout(SubLayer(X))$

In [ ]:
# 创建层归一化
class LayerNormalization(nn.Module):
    
    def __init__(self, eps: float = 10**-6) -> None: # 我们将epsilon定义为0.000001以避免除以零
        super().__init__()
        self.eps = eps
        
        # 我们将alpha定义为一个可训练参数，并将其初始化为1
        self.alpha = nn.Parameter(torch.ones(1)) # 用于缩放输入数据的一维张量
        
        # 我们将bias定义为一个可训练参数，并将其初始化为0
        self.bias = nn.Parameter(torch.zeros(1)) # 将添加到输入数据的一维张量
        
    def forward(self, x):
        mean = x.mean(dim = -1, keepdim = True) # 计算输入数据的均值。保持维度数量不变
        std = x.std(dim = -1, keepdim = True) # 计算输入数据的标准差。保持维度数量不变
        
        # 返回归一化后的输入
        return self.alpha * (x-mean) / (std + self.eps) + self.bias

PyTorch有nn.LayerNorm，但这里我们用数学公式来实现以便于学习。 

In [ ]:
# 构建残差连接
class ResidualConnection(nn.Module):
    def __init__(self, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout) # 我们使用dropout层来防止过拟合
        self.norm = LayerNormalization() # 我们使用一个归一化层
    
    def forward(self, x, sublayer):
        # 我们归一化输入并将其添加到原始输入'x'。这就创建了残差连接过程。
        return x + self.dropout(sublayer(self.norm(x))) 

### 2.4 前馈神经网络（Feedforward Networks）
**基于位置的前馈网络（Position-wise Feed-Forward Networks）**

除了注意力子层外，编码器和解码器的每一层都包含一个全连接的前馈网络，它分别且相同地应用于每个位置。这包括两个线性变换，中间有一个ReLU激活函数。

$$ FFN(x) = max(0, xW_1 + b_1)W_2 + b_2$$

虽然线性变换在不同位置是相同的，但它们在不同层之间使用不同的参数。另一种描述方式是将其视为两个核大小为1的卷积。

输入和输出的维度是$d_{model}$，内层的维度是$d_{ff}$。

### 2.5 Transformer编码器概述

现在，我们已经编写了Transformer编码器的四个部分。让我们回顾一下数据是如何通过所有这些层进行转换的。

1. **词嵌入和位置编码**：

$$X = EmbeddingLookup(X) + PositionalEncoding(X)$$

$$X \in \mathbb{R}^{batch.size \times  seq.len.\times   embed.dim.} $$

2. **自注意力和掩码**：

$$Q = Linear(X) = XW_{Q}$$
$$K = Linear(X) = XW_{K}$$
$$V = Linear(X) = XW_{V}$$
$$X_{attention} = SelfAttention(Q, K, V)$$

3. **残差连接和层归一化**：

$$X_{attention} = LayerNorm(X_{attention})$$
$$X_{attention} = X + X_{attention} $$

4. **基于位置的前馈网络**：两个带有ReLU激活函数的线性映射：
$$X_{hidden} = Linear(Activate(Linear(X_{attention})))$$

5. **重复步骤3**：
$$X_{hidden} = LayerNorm(X_{hidden})$$
$$X_{hidden} = X_{attention} + X_{hidden}$$

$$X_{hidden} \in \mathbb{R}^{batch.size \times  seq.len.\times   embed.dim.}$$

 

In [ ]:
def clones(module, N):
    """
    "生成N个相同的层。"
    使用deepcopy使权重独立。
    """
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

在论文中，$Encoder$有$N=6$个块。

In [ ]:
# 一个编码器可以有多个编码器块
class Encoder(nn.Module):
    
    # 编码器接收'EncoderBlock'的实例
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers # 存储EncoderBlocks
        self.norm = LayerNormalization() # 用于归一化编码器层输出的层
        
    def forward(self, x, mask):
        # 遍历存储在self.layers中的每个EncoderBlock
        for layer in self.layers:
            x = layer(x, mask) # 将每个EncoderBlock应用于输入张量'x'
        return self.norm(x) # 归一化输出

每个**编码器块（Encoder Block）**包含两个子层（**自注意力**、**基于位置的前馈**）和2个子层连接：
 

In [ ]:
# 构建编码器块
class EncoderBlock(nn.Module):
    
    # 此块接收MultiHeadAttentionBlock和FeedForwardBlock，以及残差连接的dropout率
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        # 存储自注意力块和前馈块
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)]) # 2个带dropout的残差连接
        
    def forward(self, x, src_mask):
        # 使用自注意力块应用第一个残差连接
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask)) # 三个'x'对应于查询、键和值输入加上源掩码
        
        # 使用前馈块应用第二个残差连接
        x = self.residual_connections[1](x, self.feed_forward_block)
        return x # 应用自注意力和前馈层以及残差连接后的输出张量

## 3. 解码器（Decoder）

在介绍编码器结构之后，我们可以看到解码器与编码器有许多相似之处。
它也是堆叠N次。但是在掩码多头注意力[0]和前馈网络[2]之间有一个编码器-解码器-上下文-注意力层（子层[1]）。它使用解码器的输出作为查询，通过多头注意力搜索编码器的输出，这使得解码器可以看到编码器的所有输出。

解码过程：
- 输入：编码输出（memory）和i-1位置的解码器输出
- 输出：i位置的输出词概率
- 解码过程像RNN一样工作

![Decoder Structure](document/images/decoder.png)

In [ ]:
# 构建解码器块
class DecoderBlock(nn.Module):
    
    # DecoderBlock接收两个MultiHeadAttentionBlock。一个是自注意力，另一个是交叉注意力。
    # 它还接收前馈块和dropout率
    def __init__(self,  self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(3)]) # 三个带dropout率的残差连接列表
        
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        
        # 使用查询、键、值以及目标语言掩码的自注意力块
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        
        # 交叉注意力块使用两个'encoder_output'作为键和值，以及源语言掩码。它还接收'x'作为解码器查询
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        
        # 带残差连接的前馈块
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x



# 构建解码器
# 一个解码器可以有多个解码器块
class Decoder(nn.Module):
    
    # 解码器接收'DecoderBlock'的实例
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        
        # 存储'DecoderBlock's
        self.layers = layers
        self.norm = LayerNormalization() # 用于归一化输出的层
        
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        
        # 遍历存储在self.layers中的每个DecoderBlock
        for layer in self.layers:
            # 将每个DecoderBlock应用于输入'x'以及编码器输出和源目标掩码
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x) # 返回归一化后的输出

我们还修改了解码器堆栈中的自注意力子层，以防止位置关注后续位置。这种掩码（**casual_mask**），加上输出嵌入偏移一个位置的事实，确保位置i的预测只能依赖于小于i位置的已知输出。


对于编码器src-mask，只需掩码padding单元格。
但对于解码器trg-mask，我们需要掩码padding并添加后续掩码过程。

In [ ]:
def casual_mask(size):
    # 创建一个维度为'size x size'的方阵，用1填充
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0

下面的注意力掩码显示了每个目标词（行）被允许查看的位置（列）。在训练期间，词被阻止关注未来的词。
"白色"表示True。

In [ ]:
subsequent_mask = casual_mask(8)

plt.figure(figsize=(5, 5))
plt.imshow(subsequent_mask[0], cmap='gray')

## 4. Transformer模型
  
最后，让我们将编码器和解码器与"生成器"结合在一起。

![](document/images/English-to-Chinese.png)

回想一下，解码器然后一次一个元素地生成一个输出序列符号。在每一步，模型都是自回归的（引用），在生成下一个时，将先前生成的符号作为额外输入。

In [ ]:
class Transformer(nn.Module):
    
    # 这需要编码器和解码器，以及源语言和目标语言的嵌入。
    # 还需要源语言和目标语言的位置编码，以及投影层
    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEncoding, tgt_pos: PositionalEncoding, projection_layer: ProjectionLayer) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer
        
    # 编码器    
    def encode(self, src, src_mask):
        src = self.src_embed(src) # 将源语言嵌入应用到输入源语言
        src = self.src_pos(src) # 将源位置编码应用到源嵌入
        return self.encoder(src, src_mask) # 返回源嵌入加上源掩码，以防止关注某些元素
    
    # 解码器
    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        tgt = self.tgt_embed(tgt) # 将目标嵌入应用到输入目标语言(tgt)
        tgt = self.tgt_pos(tgt) # 将目标位置编码应用到目标嵌入
        
        # 返回目标嵌入、编码器输出以及源和目标掩码
        # 目标掩码确保模型不会"看到"序列中的未来元素
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    # 将投影层与Softmax函数应用于解码器输出
    def project(self, x):
        return self.projection_layer(x)

In [ ]:
# 构建线性层
class ProjectionLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int) -> None: # 模型维度和输出词表的大小
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size) # 线性层，将'd_model'的特征空间投影到'vocab_size'的输出空间
    def forward(self, x):
        return torch.log_softmax(self.proj(x), dim = -1) # 对输出应用log Softmax函数

**设置参数并创建完整Transformer模型的函数**

In [ ]:
# 定义函数及其参数，包括模型维度、编码器和解码器堆叠数量、头数等
def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int = 512, N: int = 6, h: int = 8, dropout: float = 0.1, d_ff: int = 2048) -> Transformer:
    
    # 创建嵌入层
    src_embed = InputEmbeddings(d_model, src_vocab_size) # 源语言（源词表到512维向量）
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size) # 目标语言（目标词表到512维向量）
    
    # 创建位置编码层
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout) # 源语言嵌入的位置编码
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout) # 目标语言嵌入的位置编码
    
    # 创建编码器块
    encoder_blocks = [] # 初始化空的编码器块列表
    for _ in range(N): # 迭代'N'次以创建'N'个编码器块（N=6）
        encoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout) # 自注意力
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout) # 前馈网络
        
        # 将层组合成编码器块
        encoder_block = EncoderBlock(encoder_self_attention_block, feed_forward_block, dropout)
        encoder_blocks.append(encoder_block) # 将编码器块添加到编码器块列表
        
    # 创建解码器块
    decoder_blocks = [] # 初始化空的解码器块列表
    for _ in range(N): # 迭代'N'次以创建'N'个解码器块（N=6）
        decoder_self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout) # 自注意力
        decoder_cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout) # 交叉注意力
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout) # 前馈网络
        
        # 将层组合成解码器块
        decoder_block = DecoderBlock(decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
        decoder_blocks.append(decoder_block) # 将解码器块添加到解码器块列表
        
    # 使用编码器块和解码器块列表创建编码器和解码器
    encoder = Encoder(nn.ModuleList(encoder_blocks))
    decoder = Decoder(nn.ModuleList(decoder_blocks))
    
    # 创建投影层
    projection_layer = ProjectionLayer(d_model, tgt_vocab_size) # 将解码器输出映射到目标词表空间
    
    # 通过组合以上所有内容创建transformer
    transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)
    
    # 初始化参数
    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
            
    return transformer # 组装并初始化好的Transformer。准备训练和验证！

## 5. Transformer模型训练：英中翻译

正则化 **标签平滑（Label Smoothing）**

在训练期间，您可以采用值为$\epsilon_{ls}=0.1$的标签平滑（https://arxiv.org/pdf/1512.00567.pdf）。

这会损害困惑度，因为模型学会更加不确定，但会提高准确率和BLEU分数。

**损失计算（Loss Computation）**

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.).to(device)

**带Warmup的优化器学习率（Optimizer with Warmup Learning Rate）**

根据论文，他们应用了带有Adam优化器的warmup学习率，参数为$\beta_1=0.9、\beta_2=0.98$和$\epsilon = 10^{−9}$。

这将根据以下公式在训练过程中更新学习率：

$$lrate = d^{−0.5}_{model}⋅min(step\_num^{−0.5},\; step\_num⋅warmup\_steps^{−1.5})$$

这对应于在前"warmup_steps"训练步骤中线性增加学习率，之后按步数的平方根的倒数比例减少。

In [ ]:
# 这需要您自己实现
# 您可以使用pytorch中的scheduler来调整学习率

所有更新都是针对学习率的，
- model-size表示$d_{model}$。
- warmup表示warmup-steps。
- factor是一个标量。

不同模型大小和优化超参数下该模型的曲线示例。

**训练迭代器（Training Iterators）**

In [ ]:
# 训练模型
print(">>>>>>> 开始训练")
train_start = time.time()

# 初始化epoch和全局步数变量
initial_epoch = 0
global_step = 0

# 从'initial_epoch'变量迭代到config中指定的epoch数量
for epoch in range(initial_epoch, config['num_epochs']):
    # 初始化训练数据加载器的迭代器
    # 我们还使用tqdm来显示进度条
    batch_iterator = tqdm(data.train_data, desc = f'处理 epoch {epoch:02d}')
    
    # 对于每个批次...
    for batch in batch_iterator:
        model.train() # 训练模型
        
        # 将输入数据和掩码加载到GPU上
        encoder_input = batch.src.to(device)
        decoder_input = batch.tgt.to(device)
        encoder_mask = batch.src_mask.to(device)
        decoder_mask = batch.tgt_mask.to(device)
        # print(encoder_input[0], encoder_mask[0], decoder_input[0], decoder_mask[0])
        # print(encoder_input.shape, encoder_mask.shape, decoder_input.shape, decoder_mask.shape)

        # 运行张量通过Transformer
        encoder_output = model.encode(encoder_input, encoder_mask)
        decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
        proj_output = model.project(decoder_output)
        
        # 将目标标签加载到GPU上
        label = batch.tgt_y.to(device)
        
        # 计算模型输出与真实标签之间的损失
        loss = loss_fn(proj_output.view(-1, tgt_vocab_size), label.view(-1))
        
        # 更新进度条
        batch_iterator.set_postfix({f"loss": f"{loss.item():6.3f}"})
        
        # 执行反向传播
        loss.backward()
        
        # 基于梯度更新参数
        optimizer.step()
        
        # 清除梯度为下一批次做准备
        optimizer.zero_grad()
        
        global_step += 1 # 更新全局步数计数

    # 评估模型性能
    if epoch % 5 == 0:
        run_validation(model, data, data.cn_word_dict, config['seq_len'], device, lambda msg: batch_iterator.write(msg))

print(f"<<<<<<< 训练完成，耗时 {time.time()-train_start:.4f} 秒")

## 6. 使用英中翻译器进行预测

In [ ]:
# 定义获取最可能下一个标记的函数
def greedy_decode(model, source, source_mask, tokenizer_tgt, max_len, device):
    # 获取目标标记序列开始和结束的索引
    bos_id = tokenizer_tgt.get('BOS')
    eos_id = tokenizer_tgt.get('EOS')
    
    # 计算源序列的编码器输出
    encoder_output = model.encode(source, source_mask)
    # 用序列开始标记初始化解码器输入
    decoder_input = torch.empty(1,1).fill_(bos_id).type_as(source).to(device)
    
    # 循环直到达到'max_len'最大长度
    while True:
        if decoder_input.size(1) == max_len:
            break
            
        # 为解码器输入构建掩码
        decoder_mask = casual_mask(decoder_input.size(1)).type_as(source_mask).to(device)
        
        # 计算解码器的输出
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        
        # 应用投影层获取下一个标记的概率
        prob = model.project(out[:, -1])
        
        # 选择概率最高的标记
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat([decoder_input, torch.empty(1,1). type_as(source).fill_(next_word.item()).to(device)], dim=1)
        
        # 如果下一个标记是序列结束标记，则结束循环
        if next_word == eos_id:
            break
            
    return decoder_input.squeeze(0) # 解码器生成的标记序列

英中翻译示例

In [ ]:
def run_validation(model, data, tokenizer_tgt, max_len, device, print_msg, num_examples=4):
    model.eval() # 设置模型为评估模式
    count = 0 # 初始化计数器以跟踪已处理的示例数量
    
    console_width = 80 # 打印消息的固定宽度
    
    # 创建评估循环
    with torch.no_grad(): # 确保在此过程中不计算梯度
        for i, batch in enumerate(data.dev_data):
            count += 1
            encoder_input = batch.src.to(device)
            encoder_mask = batch.src_mask.to(device)
            
            # 确保验证集的batch_size为1
            assert encoder_input.size(0) ==  1, '验证集的批量大小必须为1'
            
            # 应用'greedy_decode'函数获取模型对输入批次源文本的输出
            model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_tgt, max_len, device)

            # 从批次中检索源文本和目标文本
            source_text = " ".join([data.en_index_dict[w] for w in data.dev_en[i]])
            target_text = " ".join([data.cn_index_dict[w] for w in data.dev_cn[i]])

            # 将所有内容保存在翻译列表中
            model_out_text = []
            # 转换为中文，跳过'BOS' 0
            print(model_out)
            for j in range(1, model_out.size(0)):
                sym = data.cn_index_dict[model_out[j].item()]
                if sym != 'EOS':
                    model_out_text.append(sym)
                else:
                    break

            # 打印结果
            print_msg('-'*console_width)
            print_msg(f'源文本: {source_text}')
            print_msg(f'目标文本: {target_text}')
            print_msg(f'预测结果: {model_out_text}')
            
            # 处理两个示例后，退出循环
            if count == num_examples:
                break

**英中翻译器**

In [ ]:
# 实现BLEU评分函数来测试您的模型